In [0]:
print("🗂️ CONFIGURACIÓN DE UNITY CATALOG")
print("="*70)

# Definir catálogo y schema del proyecto
CATALOG = "pandito_ds"  # Catálogo del proyecto
SCHEMA = "default"

# Crear catálogo si no existe
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
print(f"✅ Catálogo '{CATALOG}' verificado/creado")

# Crear schema si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema '{CATALOG}.{SCHEMA}' verificado/creado")

# Usar este schema por defecto
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"📌 Usando catálogo: {CATALOG}.{SCHEMA}")

print("\n📊 Tablas esperadas:")
print(f"   • Entrada: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
print(f"   • Salida:  {CATALOG}.{SCHEMA}.dl_sequences_lstm")
print(f"   • Salida:  {CATALOG}.{SCHEMA}.dl_metadata_lstm")

print("\n✅ Unity Catalog configurado")
print("="*70)

## 📚 Guía: Referencias relativas en Unity Catalog

### 🎯 Por qué usamos referencias relativas

**Problema sin UC:**
```python
# ❌ Sin Unity Catalog - problemas:
spark.table('mi_tabla')  # ¿En qué catálogo?
spark.table('default.mi_tabla')  # ¿Qué catálogo?
spark.table('hive_metastore.default.mi_tabla')  # Hardcoded, no portable
```

**Solución con UC:**
```python
# ✅ Con Unity Catalog - claro y portable:
CATALOG = "pandito_ds"
SCHEMA = "default"
TABLE = f"{CATALOG}.{SCHEMA}.mi_tabla"

# Ahora cualquier notebook puede usar:
spark.table(TABLE)  # pandito_ds.default.mi_tabla
```

**Beneficios:**
* 📦 **Portable:** Cambia el catálogo en un solo lugar
* 👥 **Colaborativo:** Todo el equipo usa las mismas referencias
* 🛡️ **Seguro:** Permisos de UC se aplican correctamente
* 📊 **Organizado:** Estructura clara de catálogos y schemas

---

### 🗂️ Estructura de Unity Catalog en este Proyecto

```
pandito_ds/                   # Catálogo del proyecto
└── default/                  # Schema por defecto
    ├── ventas_mensuales_mendoza_h3  # Tabla fuente (generada en notebook 04)
    ├── dl_sequences_lstm         # Secuencias preparadas para LSTM
    └── dl_metadata_lstm          # Metadatos y scaler
```

**Convención de nombres:**
* Catálogo: `pandito_ds` (Data Science del proyecto Pandito)
* Schema: `default` (podemos crear otros: `staging`, `production`)
* Tablas: nombres descriptivos en snake_case

---

### 💻 Patrón de uso en todos los notebooks

**Al inicio de cada notebook:**
```python
# 1. Definir referencias
CATALOG = "pandito_ds"
SCHEMA = "default"

# 2. Usar el catálogo
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# 3. Leer tablas con referencia completa (RECOMENDADO)
df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")

# O con referencia relativa (después de USE CATALOG/SCHEMA)
df = spark.table("ventas_mensuales_mendoza_h3")
```

**Al guardar tablas:**
```python
# SIEMPRE usar referencia completa al guardar
TABLE_NAME = f"{CATALOG}.{SCHEMA}.nueva_tabla"
df.write.mode('overwrite').saveAsTable(TABLE_NAME)
```

---

### ⚡ Migración desde Hive Metastore

Si tienes tablas antiguas en `hive_metastore`:

```python
# Leer de hive_metastore
df_old = spark.table("hive_metastore.default.tabla_vieja")

# Guardar en Unity Catalog
df_old.write.mode('overwrite').saveAsTable("pandito_ds.default.tabla_nueva")

print("✅ Tabla migrada a Unity Catalog")
```

---

### 🔐 Permisos y seguridad

Unity Catalog controla permisos a nivel de:
* **Catálogo:** `GRANT USE CATALOG ON pandito_ds TO user@domain.com`
* **Schema:** `GRANT USE SCHEMA ON pandito_ds.default TO user@domain.com`
* **Tabla:** `GRANT SELECT ON pandito_ds.default.mi_tabla TO user@domain.com`

En Databricks Free Edition, el creador tiene todos los permisos.

---

### 💡 Tips importantes

**✅ Buenas Prácticas:**
* Definir `CATALOG` y `SCHEMA` como constantes al inicio
* Usar referencias completas al guardar
* Documentar la estructura en cada notebook
* Crear schemas separados para staging/production

**❌ Evitar:**
* Hardcodear nombres de tablas sin variables
* Mezclar tablas de diferentes catálogos sin claridad
* Usar `default.mi_tabla` sin especificar el catálogo
* Asumir que otros conocen la estructura de tu catálogo

In [0]:
# Instalar H3 para trabajar con datos geoespaciales
!pip install h3 --quiet

In [0]:
import pandas as pd
import numpy as np
import h3
from datetime import datetime, timedelta

print("🏭 GENERANDO DATOS SINTÉTICOS: LOS ANDES MARKET")
print("="*70)

# Configuración
np.random.seed(42)

# Definir sucursales con georeferenciación real en Mendoza
sucursales = [
    {
        'sucursal_id': 'SUC001',
        'sucursal_nombre': 'Centro - San Martín',
        'lat': -32.8895,
        'lon': -68.8458,
        'zona': 'Centro Comercial'
    },
    {
        'sucursal_id': 'SUC002',
        'sucursal_nombre': 'Las Heras',
        'lat': -32.8507,
        'lon': -68.8269,
        'zona': 'Zona Residencial'
    },
    {
        'sucursal_id': 'SUC003',
        'sucursal_nombre': 'Guaymallén - Av. San Martín',
        'lat': -32.9089,
        'lon': -68.7878,
        'zona': 'Corredor Comercial'
    },
    {
        'sucursal_id': 'SUC004',
        'sucursal_nombre': 'Godoy Cruz - Av. San Francisco',
        'lat': -32.9226,
        'lon': -68.8440,
        'zona': 'Zona Comercial'
    },
    {
        'sucursal_id': 'SUC005',
        'sucursal_nombre': 'Maipú - Rodríguez Peña',
        'lat': -32.9815,
        'lon': -68.7920,
        'zona': 'Zona Residencial'
    }
]

# Generar 60 meses de datos (2019-2024)
fecha_inicio = datetime(2019, 1, 1)
fechas = [fecha_inicio + timedelta(days=30*i) for i in range(60)]

print(f"📅 Período: {fechas[0].date()} a {fechas[-1].date()}")
print(f"🏪 Sucursales: {len(sucursales)}")
print(f"📊 Registros a generar: {len(fechas) * len(sucursales)}")

# Generar datos
records = []

for fecha in fechas:
    mes = fecha.month
    año = fecha.year
    
    for suc in sucursales:
        # Ventas base según ubicación
        if suc['zona'] == 'Centro Comercial':
            base_ventas = 150000
        elif suc['zona'] == 'Corredor Comercial':
            base_ventas = 120000
        else:
            base_ventas = 80000
        
        # Tendencia creciente
        tendencia = (año - 2019) * 10000 + (mes - 1) * 1000
        
        # Estacionalidad (pico en verano: dic-feb)
        estacionalidad = 20000 * np.sin(2 * np.pi * mes / 12 + np.pi/2)
        
        # Ruido aleatorio
        ruido = np.random.normal(0, 5000)
        
        ventas = max(base_ventas + tendencia + estacionalidad + ruido, 10000)
        
        # Indexación H3
        h3_res9 = h3.latlng_to_cell(suc['lat'], suc['lon'], 9)  # ~174m
        h3_res8 = h3.latlng_to_cell(suc['lat'], suc['lon'], 8)  # ~461m
        h3_res7 = h3.latlng_to_cell(suc['lat'], suc['lon'], 7)  # ~1.22km
        
        record = {
            'fecha': fecha,
            'sucursal_id': suc['sucursal_id'],
            'sucursal_nombre': suc['sucursal_nombre'],
            'lat': suc['lat'],
            'lon': suc['lon'],
            'zona': suc['zona'],
            'ventas': round(ventas, 2),
            'h3_index': h3_res9,
            'h3_res8': h3_res8,
            'h3_res7': h3_res7
        }
        records.append(record)

df_ventas = pd.DataFrame(records)

print("\n✅ Datos sintéticos generados")
print(f"   Total registros: {len(df_ventas):,}")
print(f"\n📊 Muestra de datos:")
display(df_ventas.head(10))

print("\n📈 Estadísticas de ventas:")
print(df_ventas.groupby('sucursal_id')['ventas'].describe())

# Guardar en Unity Catalog
CATALOG = "pandito_ds"
SCHEMA = "default"
TABLE_BASE = f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3"

print(f"\n💾 Guardando en Unity Catalog...")
print(f"   Tabla: {TABLE_BASE}")

df_spark_ventas = spark.createDataFrame(df_ventas)
df_spark_ventas.write.mode('overwrite').saveAsTable(TABLE_BASE)

print(f"\n✅ Tabla '{TABLE_BASE}' creada")
print("="*70)
print("\n🎯 TABLA BASE GENERADA - LISTA PARA USO EN TODO EL PROYECTO")
print(f"\n   Acceder desde cualquier notebook con:")
print(f"   df = spark.table('{TABLE_BASE}').toPandas()")
print("="*70)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h3
from pyspark.sql import SparkSession
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Librerías importadas correctamente")
print(f"   H3 versión: {h3.__version__}")

## 1️⃣ Cargar datos desde Unity Catalog

### 📊 Dataset: Los Andes Market

Cargaremos los datos georeferenciados generados en el **Notebook 04**.

**Tabla fuente:** `pandito_ds.default.ventas_mensuales_mendoza_h3`

**Contenido:**
* 📅 **Período:** 2019-2024 (60 meses)
* 🏪 **Sucursales:** 5 ubicaciones en Mendoza
* 📍 **Georeferenciación:** Coordenadas GPS + índices H3
* 💰 **Métrica:** Ventas mensuales por sucursal
* 📈 **Registros:** 300 (60 meses × 5 sucursales)

**Columnas clave:**
* `fecha`, `sucursal_id`, `ventas`
* `lat`, `lon` (coordenadas GPS)
* `h3_index`, `h3_res8`, `h3_res7` (indexación hexagonal)
* `zona` (tipo de zona comercial)

In [0]:
# Inicializar Spark
spark = SparkSession.builder.getOrCreate()

# Configurar referencias relativas
CATALOG = "pandito_ds"
SCHEMA = "default"
TABLE_NAME = "ventas_mensuales_mendoza_h3"

# Usar referencia relativa de Unity Catalog
full_table_name = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(f"📂 Cargando datos desde Unity Catalog...")
print(f"   Tabla: {full_table_name}")

# Cargar datos georeferenciados
df_spark = spark.table(full_table_name)
df = df_spark.toPandas()
df = df.sort_values(['sucursal_id', 'fecha']).reset_index(drop=True)

print("📈 DATOS GEOREFERENCIADOS CARGADOS:")
print(f"   Registros: {len(df):,}")
print(f"   Sucursales: {df['sucursal_id'].nunique()}")
print(f"   Período: {df['fecha'].min()} a {df['fecha'].max()}")
print(f"\n🏪 Sucursales:")
for suc_id in df['sucursal_id'].unique():
    print(f"   - {suc_id}: {df[df['sucursal_id']==suc_id]['sucursal_nombre'].iloc[0]}")

print(f"\n🔍 Primeras filas:")
display(df.head(10))

print(f"\n📈 Estadísticas de ventas:")
print(df['ventas'].describe())

print(f"\n🗺️ Campos geográficos disponibles:")
print(f"   - lat, lon: Coordenadas GPS")
print(f"   - h3_index: Índice H3 resolución 9 (~174m)")
print(f"   - h3_res8: Índice H3 resolución 8 (~461m)")
print(f"   - h3_res7: Índice H3 resolución 7 (~1.22km)")
print(f"   - zona: Tipo de zona geográfica")

## 2️⃣ Feature Engineering: Temporal + Geoespacial

### 🧠 Por qué Feature Engineering

Las redes neuronales aprenden patrones, pero **necesitamos ayudarlas** a ver lo importante:
* 📅 **Temporales:** Tendencias, estacionalidad, momentum
* 📍 **Geoespaciales:** Ubicación, densidad, competencia

---

### 📅 Features Temporales

**1. Básicos:**
* `mes` (1-12), `trimestre` (1-4), `dia_año` (1-365)

**2. Cíclicos (sin/cos):**
* `mes_sin`, `mes_cos` → Evita discontinuidad diciembre-enero
* `trimestre_sin`, `trimestre_cos`

**3. Lags (valores pasados) POR SUCURSAL:**
* `ventas_lag_1`, `ventas_lag_2`, `ventas_lag_3`
* `ventas_lag_6`, `ventas_lag_12`
* ⚠️ **IMPORTANTE:** Se calculan por sucursal (no mezclar datos)

**4. Rolling Statistics POR SUCURSAL:**
* `ventas_rolling_mean_3`, `ventas_rolling_mean_6`, `ventas_rolling_mean_12`
* `ventas_rolling_std_3`, `ventas_rolling_std_6`, `ventas_rolling_std_12`

**5. Tasa de Cambio:**
* `ventas_pct_change` → Growth rate mensual

---

### 📍 Features Geoespaciales (H3)

**1. Distancia al Centro:**
* `distancia_centro_km` → Distancia desde sucursal a Plaza Independencia (centro de Mendoza)
* Cálculo con fórmula Haversine (distancia entre coordenadas GPS)

**2. Densidad H3:**
* `densidad_h3` → Número de sucursales en hexágonos vecinos
* Usa H3 resolución 8 (~461m por hexágono)

**3. Zona Comercial:**
* One-hot encoding: `zona_Centro Comercial`, `zona_Corredor Comercial`, etc.

---

### 💡 Por qué estas Features

* **Lags:** Capturan dependencia temporal ("mes pasado influye en este mes")
* **Rolling:** Suavizan ruido y revelan tendencias
* **Cíclicos:** Capturan periodicidad sin discontinuidades
* **Geoespaciales:** Ubicación impacta ventas (centro vs periferia, densidad de competencia)

In [0]:
from math import radians, cos, sin, asin, sqrt

def haversine(lat1, lon1, lat2, lon2):
    """
    Calcular distancia en km entre dos puntos GPS.
    """
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    km = 6371 * c
    return km

# Asegurar que fecha es datetime
df['fecha'] = pd.to_datetime(df['fecha'])

# Features temporales básicas
df['mes'] = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter
df['dia_año'] = df['fecha'].dt.dayofyear

# Features cíclicos (capturan la naturaleza circular del tiempo)
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
df['trimestre_sin'] = np.sin(2 * np.pi * df['trimestre'] / 4)
df['trimestre_cos'] = np.cos(2 * np.pi * df['trimestre'] / 4)

# IMPORTANTE: Lags y rolling features POR SUCURSAL
# (no mezclar datos entre sucursales)
print("⏳ Calculando lags y rolling features por sucursal...")

for suc_id in df['sucursal_id'].unique():
    mask = df['sucursal_id'] == suc_id
    
    # Lags (valores pasados)
    for i in [1, 2, 3, 6, 12]:
        df.loc[mask, f'ventas_lag_{i}'] = df.loc[mask, 'ventas'].shift(i)
    
    # Rolling means (promedios móviles)
    for window in [3, 6, 12]:
        df.loc[mask, f'ventas_rolling_mean_{window}'] = df.loc[mask, 'ventas'].rolling(window=window).mean()
        df.loc[mask, f'ventas_rolling_std_{window}'] = df.loc[mask, 'ventas'].rolling(window=window).std()
    
    # Tasa de cambio (growth rate)
    df.loc[mask, 'ventas_pct_change'] = df.loc[mask, 'ventas'].pct_change()

print("✅ Features temporales creadas por sucursal")

# Features geográficas
print("\n🗺️ Calculando features geográficas H3...")

# Centro de Mendoza (Plaza Independencia)
CENTRO_MENDOZA_LAT = -32.8895
CENTRO_MENDOZA_LON = -68.8458

# Distancia al centro
df['distancia_centro_km'] = df.apply(
    lambda row: haversine(row['lat'], row['lon'], CENTRO_MENDOZA_LAT, CENTRO_MENDOZA_LON),
    axis=1
)

# Densidad H3: contar sucursales en hexágonos vecinos (resolución 8)
# Obtener vecinos de cada hexágono
for idx, row in df.iterrows():
    h3_index = row['h3_res8']
    vecinos = h3.grid_disk(h3_index, 1)  # Hexágono + vecinos inmediatos
    # Contar cuántas sucursales caen en estos hexágonos
    n_sucursales_vecinas = df[df['h3_res8'].isin(vecinos)]['sucursal_id'].nunique()
    df.at[idx, 'densidad_h3'] = n_sucursales_vecinas

# One-hot encoding de zona
df_zona_encoded = pd.get_dummies(df['zona'], prefix='zona')
df = pd.concat([df, df_zona_encoded], axis=1)

print("✅ Features geográficas creadas")
print(f"   - Distancia al centro (km)")
print(f"   - Densidad H3 (sucursales vecinas)")
print(f"   - One-hot zona: {list(df_zona_encoded.columns)}")

# Eliminar filas con NaN (causadas por lags y rolling)
df_clean = df.dropna().reset_index(drop=True)

print(f"\n✅ FEATURES COMPLETOS:")
print(f"   Total features: {df_clean.shape[1]}")
print(f"   Registros después de limpieza: {len(df_clean):,} (se eliminaron {len(df) - len(df_clean)} por NaN)")

print(f"\n📄 CATEGORÍAS DE FEATURES:")
print(f"   Temporales básicos: mes, trimestre, día_año")
print(f"   Cíclicos: mes_sin/cos, trimestre_sin/cos")
print(f"   Lags: ventas_lag_1,2,3,6,12")
print(f"   Rolling: ventas_rolling_mean/std_3,6,12")
print(f"   Tasa de cambio: ventas_pct_change")
print(f"   Geográficos: distancia_centro_km, densidad_h3, zona_*")

In [0]:
# Visualizar features
fig, axes = plt.subplots(3, 2, figsize=(16, 14))

# Seleccionar una sucursal para visualizar (SUC001 - Centro)
suc_ejemplo = 'SUC001'
df_ejemplo = df_clean[df_clean['sucursal_id'] == suc_ejemplo].sort_values('fecha')

# 1. Ventas originales con lags
axes[0, 0].plot(df_ejemplo['fecha'], df_ejemplo['ventas'], label='Ventas', linewidth=2.5, color='#2E86AB', marker='o')
axes[0, 0].plot(df_ejemplo['fecha'], df_ejemplo['ventas_lag_1'], label='Lag 1 mes', linewidth=1.5, alpha=0.7, color='#A23B72', linestyle='--')
axes[0, 0].plot(df_ejemplo['fecha'], df_ejemplo['ventas_lag_3'], label='Lag 3 meses', linewidth=1.5, alpha=0.7, color='#F18F01', linestyle='--')
axes[0, 0].set_title(f'🔄 Ventas con Lags Temporales ({suc_ejemplo})', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Ventas ($)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Promedios móviles
axes[0, 1].plot(df_ejemplo['fecha'], df_ejemplo['ventas'], label='Ventas', linewidth=2, alpha=0.4, color='#2E86AB')
axes[0, 1].plot(df_ejemplo['fecha'], df_ejemplo['ventas_rolling_mean_3'], label='Media Móvil 3m', linewidth=2.5, color='#6A994E')
axes[0, 1].plot(df_ejemplo['fecha'], df_ejemplo['ventas_rolling_mean_6'], label='Media Móvil 6m', linewidth=2.5, color='#F18F01')
axes[0, 1].set_title(f'📉 Promedios Móviles ({suc_ejemplo})', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Ventas ($)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Tasa de cambio
axes[1, 0].plot(df_ejemplo['fecha'], df_ejemplo['ventas_pct_change'] * 100, linewidth=2, color='#A23B72', marker='o')
axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1, 0].fill_between(df_ejemplo['fecha'], 0, df_ejemplo['ventas_pct_change'] * 100, 
                        where=(df_ejemplo['ventas_pct_change'] > 0), alpha=0.3, color='green')
axes[1, 0].fill_between(df_ejemplo['fecha'], 0, df_ejemplo['ventas_pct_change'] * 100, 
                        where=(df_ejemplo['ventas_pct_change'] < 0), alpha=0.3, color='red')
axes[1, 0].set_title(f'📈 Tasa de Crecimiento Mensual ({suc_ejemplo})', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Cambio (%)')
axes[1, 0].set_xlabel('Fecha')
axes[1, 0].grid(True, alpha=0.3)

# 4. Features cíclicos
axes[1, 1].scatter(df_clean['mes_sin'], df_clean['mes_cos'], c=df_clean['mes'], cmap='hsv', s=50, alpha=0.6, edgecolor='black')
axes[1, 1].set_title('🔄 Features Cíclicos (Sin/Cos del Mes)', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('sin(mes)')
axes[1, 1].set_ylabel('cos(mes)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_aspect('equal')

# 5. Distancia al centro vs Ventas promedio
ventas_por_suc = df_clean.groupby('sucursal_id').agg({
    'ventas': 'mean',
    'distancia_centro_km': 'first',
    'sucursal_nombre': 'first'
}).reset_index()

axes[2, 0].scatter(ventas_por_suc['distancia_centro_km'], ventas_por_suc['ventas'], 
                  s=200, alpha=0.7, color='#2E86AB', edgecolor='black', linewidth=2)
for _, row in ventas_por_suc.iterrows():
    axes[2, 0].annotate(row['sucursal_id'], 
                       (row['distancia_centro_km'], row['ventas']),
                       fontsize=9, ha='center', va='center', fontweight='bold', color='white')
axes[2, 0].set_title('🗺️ Distancia al Centro vs Ventas Promedio', fontsize=13, fontweight='bold')
axes[2, 0].set_xlabel('Distancia al Centro (km)')
axes[2, 0].set_ylabel('Ventas Promedio ($)')
axes[2, 0].grid(True, alpha=0.3)

# 6. Densidad H3 por sucursal
densidad_por_suc = df_clean.groupby('sucursal_id').agg({
    'densidad_h3': 'first',
    'ventas': 'mean'
}).reset_index()

colores_bar = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261']
barras = axes[2, 1].bar(densidad_por_suc['sucursal_id'], densidad_por_suc['densidad_h3'], 
                        color=colores_bar, alpha=0.8, edgecolor='black', linewidth=2)
axes[2, 1].set_title('🏪 Densidad H3: Sucursales Vecinas', fontsize=13, fontweight='bold')
axes[2, 1].set_xlabel('Sucursal')
axes[2, 1].set_ylabel('Número de Sucursales Vecinas')
axes[2, 1].grid(axis='y', alpha=0.3)

for barra, dens in zip(barras, densidad_por_suc['densidad_h3']):
    axes[2, 1].text(barra.get_x() + barra.get_width()/2., dens,
                   f'{int(dens)}',
                   ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🔍 INSIGHTS:")
print(f"   • Features cíclicos (sin/cos) evitan discontinuidades (ej: diciembre -> enero)")
print(f"   • Lags y rolling features suavizan ruido y capturan tendencias")
print(f"   • Distancia al centro puede correlacionar con volumen de ventas")
print(f"   • Densidad H3 indica concentración de sucursales (competencia o sinergias)")

## 3️⃣ Normalización para Deep Learning

### 🧠 Por qué normalizar

Las redes neuronales funcionan **mucho mejor** con datos normalizados:

**Sin normalizar:**
* `ventas`: [10,000 - 500,000] → Gradientes grandes, inestabilidad
* `mes`: [1 - 12] → Diferentes escalas
* `distancia_km`: [0.5 - 15] → Magnitudes distintas

**Con MinMaxScaler [0, 1]:**
* Todos los features en la misma escala
* Convergencia más rápida
* Evita que features con valores grandes dominen

---

### 🛡️ Regla de Oro: Evitar Data Leakage

**⚠️ IMPORTANTE - NO HACER ESTO:**
```python
# ❌ MAL: Ajustar con todos los datos
scaler.fit(df_completo)
```

**✅ CORRECTO:**
```python
# 1. Ajustar SOLO con training
scaler.fit(train_data)

# 2. Transformar train, val, test con el MISMO scaler
train_scaled = scaler.transform(train_data)
val_scaled = scaler.transform(val_data)
test_scaled = scaler.transform(test_data)
```

**Por qué:** Si el scaler "ve" los datos de test, el modelo tendría información del futuro.

---

### 💾 Guardar el Scaler

El scaler se **serializa y guarda en Unity Catalog** para:
* Normalizar nuevos datos en producción
* Desnormalizar predicciones (volver a escala original)
* Reproducibilidad

In [0]:
# División temporal (respetando orden cronológico)
# 70% train, 15% validation, 15% test

n_total = len(df_clean)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_data = df_clean.iloc[:n_train].copy()
val_data = df_clean.iloc[n_train:n_train+n_val].copy()
test_data = df_clean.iloc[n_train+n_val:].copy()

print("📏 DIVISIÓN DE DATOS (Temporal)")
print("="*70)
print(f"TRAIN: {len(train_data):3d} meses | {train_data['fecha'].min().strftime('%Y-%m')} a {train_data['fecha'].max().strftime('%Y-%m')}")
print(f"VAL:   {len(val_data):3d} meses | {val_data['fecha'].min().strftime('%Y-%m')} a {val_data['fecha'].max().strftime('%Y-%m')}")
print(f"TEST:  {len(test_data):3d} meses | {test_data['fecha'].min().strftime('%Y-%m')} a {test_data['fecha'].max().strftime('%Y-%m')}")
print("="*70)

# Visualizar división
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(train_data['fecha'], train_data['ventas'], label='Train', linewidth=2, color='#2E86AB')
ax.plot(val_data['fecha'], val_data['ventas'], label='Validation', linewidth=2, color='#F18F01')
ax.plot(test_data['fecha'], test_data['ventas'], label='Test', linewidth=2, color='#A23B72')

ax.set_title('División Temporal de Datos', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Ventas ($)')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Seleccionar features para normalizar (excluir columnas categóricas y de identificación)
columnas_excluir = ['fecha', 'sucursal_id', 'sucursal_nombre', 'zona', 'h3_index', 'h3_res8', 'h3_res7']
features_to_scale = [col for col in df_clean.columns if col not in columnas_excluir]

# Inicializar scalers
scaler = MinMaxScaler(feature_range=(0, 1))

# Ajustar scaler SOLO con datos de entrenamiento
scaler.fit(train_data[features_to_scale])

# Transformar todos los conjuntos
train_scaled = scaler.transform(train_data[features_to_scale])
val_scaled = scaler.transform(val_data[features_to_scale])
test_scaled = scaler.transform(test_data[features_to_scale])

# Convertir a DataFrames para visualización
train_scaled_df = pd.DataFrame(train_scaled, columns=features_to_scale)
val_scaled_df = pd.DataFrame(val_scaled, columns=features_to_scale)
test_scaled_df = pd.DataFrame(test_scaled, columns=features_to_scale)

print("✅ Datos normalizados exitosamente")
print(f"\n📊 Estadísticas de 'ventas' normalizadas (Train):")
print(f"   Mínimo: {train_scaled_df['ventas'].min():.4f}")
print(f"   Máximo: {train_scaled_df['ventas'].max():.4f}")
print(f"   Media:   {train_scaled_df['ventas'].mean():.4f}")

# Visualizar normalización
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Antes de normalizar
axes[0].hist(train_data['ventas'], bins=30, color='#2E86AB', alpha=0.7, edgecolor='black')
axes[0].set_title('Distribución ANTES de Normalizar', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Ventas ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(True, alpha=0.3)

# Después de normalizar
axes[1].hist(train_scaled_df['ventas'], bins=30, color='#6A994E', alpha=0.7, edgecolor='black')
axes[1].set_title('Distribución DESPUÉS de Normalizar [0, 1]', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Ventas Normalizadas')
axes[1].set_ylabel('Frecuencia')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
def create_sequences(data, target_col_idx=0, lookback=12, forecast_horizon=1):
    """
    Crea secuencias para LSTM.
    
    Args:
        data: array numpy normalizado
        target_col_idx: índice de la columna objetivo en data
        lookback: número de pasos temporales pasados
        forecast_horizon: número de pasos futuros a predecir
    
    Returns:
        X: secuencias de entrada (samples, lookback, features)
        y: valores objetivo (samples, forecast_horizon)
    """
    X, y = [], []
    
    for i in range(lookback, len(data) - forecast_horizon + 1):
        # Secuencia de entrada: [i-lookback:i, todas las features]
        X.append(data[i-lookback:i, :])
        
        # Objetivo: valor futuro de la columna target
        if forecast_horizon == 1:
            y.append(data[i, target_col_idx])
        else:
            y.append(data[i:i+forecast_horizon, target_col_idx])
    
    return np.array(X), np.array(y)

print("✅ Función create_sequences definida")

In [0]:
# Parámetros
LOOKBACK = 12  # Usar últimos 12 meses para predecir
FORECAST_HORIZON = 1  # Predecir 1 mes adelante

# Crear secuencias para cada conjunto
X_train, y_train = create_sequences(train_scaled, target_col_idx=0, lookback=LOOKBACK, forecast_horizon=FORECAST_HORIZON)
X_val, y_val = create_sequences(val_scaled, target_col_idx=0, lookback=LOOKBACK, forecast_horizon=FORECAST_HORIZON)
X_test, y_test = create_sequences(test_scaled, target_col_idx=0, lookback=LOOKBACK, forecast_horizon=FORECAST_HORIZON)

print("🔢 DIMENSIONES DE LOS DATOS")
print("="*70)
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape} | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape} | y_test:  {y_test.shape}")
print("="*70)

print(f"\n📊 Interpretación:")
print(f"   - {X_train.shape[0]} secuencias de entrenamiento")
print(f"   - Cada secuencia tiene {X_train.shape[1]} pasos temporales (meses)")
print(f"   - Cada paso tiene {X_train.shape[2]} features")
print(f"   - Predecir {FORECAST_HORIZON} mes(es) adelante")

# Visualizar una secuencia de ejemplo
fig, ax = plt.subplots(figsize=(14, 5))

ejemplo_idx = 0
secuencia_ejemplo = X_train[ejemplo_idx, :, 0]  # Primera feature (ventas normalizadas)
target_ejemplo = y_train[ejemplo_idx]

tiempos = np.arange(1, LOOKBACK + 1)
ax.plot(tiempos, secuencia_ejemplo, 'o-', linewidth=2, markersize=8, color='#2E86AB', label='Secuencia de entrada (12 meses)')
ax.plot(LOOKBACK + 1, target_ejemplo, 'r*', markersize=20, label=f'Target a predecir (mes {LOOKBACK+1})')

ax.set_title('🔍 Ejemplo de Secuencia para LSTM (Ventas Normalizadas)', fontsize=14, fontweight='bold')
ax.set_xlabel('Paso Temporal (Mes Relativo)', fontsize=12)
ax.set_ylabel('Ventas Normalizadas [0, 1]', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, LOOKBACK + 2))

plt.tight_layout()
plt.show()

print(f"\n💡 El modelo LSTM verá los {LOOKBACK} meses anteriores y aprenderá a predecir el siguiente")

## 5️⃣ Guardar datos en Unity Catalog

### 💾 Estrategia de Persistencia

Guardaremos los datos procesados en **Unity Catalog** para:

✅ **Acceso desde cualquier notebook**  
✅ **Persistencia permanente** (no se borra al reiniciar cluster)  
✅ **Control de versiones** (Delta Lake)  
✅ **Seguridad** (permisos de UC)  
✅ **Reproducibilidad** (otros pueden usar los mismos datos)

---

### 📊 Tablas a crear

**1. `pandito_ds.default.dl_sequences_lstm`**
* Todas las secuencias preparadas (train/val/test)
* Serializadas en base64 para preservar estructura 3D

**2. `pandito_ds.default.dl_metadata_lstm`**
* Parámetros: lookback, forecast_horizon, n_features
* Nombres de features
* **Scaler serializado** (para desnormalizar predicciones)

---

### 💻 Cómo Deserializar

Los próximos notebooks cargarán así:

```python
import pickle, base64, numpy as np

# Leer y deserializar secuencias
df = spark.table('pandito_ds.default.dl_sequences_lstm').toPandas()
train_df = df[df['split'] == 'train']

X_train = np.array([
    pickle.loads(base64.b64decode(row['sequence_data_b64']))
    for _, row in train_df.iterrows()
])
```

In [0]:
import pickle
import os
import base64
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, ArrayType, TimestampType

# ============================================================================
# OPCIÓN 1: Guardar en Delta Lake (PERSISTENTE, accesible desde cualquier notebook)
# ============================================================================

print("💾 Guardando datos en Delta Lake...\n")

# Función auxiliar para convertir array 3D a DataFrame
# Usa serialización binaria (pickle + base64) para preservar la estructura 3D
def sequences_to_dataframe(X, y, split_name, sequence_offset=0):
    """
    Convierte secuencias 3D en DataFrame con una fila por secuencia.
    Cada secuencia se serializa como string base64 para preservar la estructura exacta.
    
    Args:
        X: array 3D de secuencias (samples, timesteps, features)
        y: array 1D de targets
        split_name: 'train', 'validation', o 'test'
        sequence_offset: offset para el sequence_id (para mantener único)
    """
    records = []
    for i in range(len(X)):
        # Serializar la secuencia 2D (timesteps x features) como bytes base64
        seq_bytes = pickle.dumps(X[i])
        seq_b64 = base64.b64encode(seq_bytes).decode('utf-8')
        
        record = {
            'sequence_id': sequence_offset + i,
            'split': split_name,
            'target_value': float(y[i]),
            # Guardar la secuencia serializada
            'sequence_data_b64': seq_b64,
            # También guardar shape para verificación
            'timesteps': int(X[i].shape[0]),
            'features': int(X[i].shape[1])
        }
        records.append(record)
    return pd.DataFrame(records)

# Convertir a DataFrames
df_train = sequences_to_dataframe(X_train, y_train, 'train', sequence_offset=0)
df_val = sequences_to_dataframe(X_val, y_val, 'validation', sequence_offset=len(X_train))
df_test = sequences_to_dataframe(X_test, y_test, 'test', sequence_offset=len(X_train) + len(X_val))

# Combinar todos los splits en una sola tabla
df_all_sequences = pd.concat([df_train, df_val, df_test], ignore_index=True)

# Convertir a Spark DataFrame y guardar
sparkdf_sequences = spark.createDataFrame(df_all_sequences)
# Configurar referencias relativas de Unity Catalog
CATALOG = "pandito_ds"
SCHEMA = "default"
TABLE_SEQUENCES = f"{CATALOG}.{SCHEMA}.dl_sequences_lstm"
TABLE_METADATA = f"{CATALOG}.{SCHEMA}.dl_metadata_lstm"

print("💾 GUARDANDO EN UNITY CATALOG")
print("="*70)
print(f"Catálogo: {CATALOG}")
print(f"Schema:   {SCHEMA}")
print("="*70)

# Guardar con referencia completa a Unity Catalog
sparkdf_sequences.write.mode('overwrite').saveAsTable(TABLE_SEQUENCES)

print(f"\n✅ Tabla '{TABLE_SEQUENCES}' creada:")
print(f"   • Train: {len(df_train)} secuencias")
print(f"   • Validation: {len(df_val)} secuencias")
print(f"   • Test: {len(df_test)} secuencias")
print(f"   • Total: {len(df_all_sequences)} secuencias")

# Guardar metadatos (incluye scaler serializado)
scaler_bytes = pickle.dumps(scaler)
scaler_b64 = base64.b64encode(scaler_bytes).decode('utf-8')

metadata_records = [{
    'param_name': 'lookback',
    'param_value': str(LOOKBACK),
    'param_type': 'int'
}, {
    'param_name': 'forecast_horizon',
    'param_value': str(FORECAST_HORIZON),
    'param_type': 'int'
}, {
    'param_name': 'n_features',
    'param_value': str(X_train.shape[2]),
    'param_type': 'int'
}, {
    'param_name': 'feature_names',
    'param_value': ','.join(features_to_scale),
    'param_type': 'list'
}, {
    'param_name': 'target_feature',
    'param_value': 'ventas',
    'param_type': 'str'
}, {
    'param_name': 'scaler_minmax',
    'param_value': scaler_b64,
    'param_type': 'pickled_base64'
}]

df_metadata = pd.DataFrame(metadata_records)
sparkdf_metadata = spark.createDataFrame(df_metadata)
# Guardar metadata con referencia completa a Unity Catalog
sparkdf_metadata.write.mode('overwrite').saveAsTable(TABLE_METADATA)

print(f"\n✅ Tabla '{TABLE_METADATA}' creada con:")
print(f"   • Parámetros del modelo")
print(f"   • Nombres de features")
print(f"   • Scaler serializado")

# ============================================================================
# OPCIÓN 2: También guardar en /tmp como respaldo local (para debug rápido)
# ============================================================================

DATA_DIR = '/tmp/dl_data'
os.makedirs(DATA_DIR, exist_ok=True)

# Guardar arrays numpy
np.save(os.path.join(DATA_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(DATA_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(DATA_DIR, 'X_val.npy'), X_val)
np.save(os.path.join(DATA_DIR, 'y_val.npy'), y_val)
np.save(os.path.join(DATA_DIR, 'X_test.npy'), X_test)
np.save(os.path.join(DATA_DIR, 'y_test.npy'), y_test)

# Guardar el scaler
with open(os.path.join(DATA_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# Guardar metadatos
metadata = {
    'lookback': LOOKBACK,
    'forecast_horizon': FORECAST_HORIZON,
    'n_features': X_train.shape[2],
    'feature_names': features_to_scale,
    'target_feature': 'ventas'
}

with open(os.path.join(DATA_DIR, 'metadata.pkl'), 'wb') as f:
    pickle.dump(metadata, f)

print(f"\n✅ Respaldo guardado en {DATA_DIR}")

print("\n" + "="*70)
print("📊 RESUMEN DE EXPORTACIÓN")
print("="*70)
print("\n🔵 UNITY CATALOG (Persistente - RECOMENDADO):")
print(f"   • {TABLE_SEQUENCES}")
print(f"   • {TABLE_METADATA}")
print("\n   💡 Cómo acceder desde otros notebooks:")
print("")
print("   # Método 1: Referencia completa (RECOMENDADO)")
print(f"   df = spark.table('{TABLE_SEQUENCES}')")
print("")
print("   # Método 2: Referencia relativa (después de USE CATALOG)")
print(f"   spark.sql('USE CATALOG {CATALOG}')")
print(f"   df = spark.table('{SCHEMA}.dl_sequences_lstm')")
print("\n🟡 /TMP (Local - Solo para debug):")
print(f"   • {DATA_DIR}")
print("   ⚠️  Se borra al reiniciar el cluster")
print("="*70)

## 🎓 Conclusiones del notebook

### ✅ Lo que lograste

**1. Feature Engineering completo:**
   * ✅ Features temporales: lags (1, 2, 3, 6, 12 meses)
   * ✅ Rolling statistics: medias y desviaciones móviles (3, 6, 12 meses)
   * ✅ Features cíclicos: sin/cos para capturar periodicidad
   * ✅ Tasa de crecimiento: pct_change mensual
   * ✅ Features geoespaciales: distancia al centro, densidad H3

**2. Preparación para Deep Learning:**
   * ✅ Normalización MinMaxScaler [0, 1]
   * ✅ Secuencias temporales para LSTM (lookback=12 meses)
   * ✅ Formato 3D: (samples, timesteps, features)
   * ✅ División temporal: 70% train, 15% val, 15% test
   * ✅ Sin data leakage (orden cronológico respetado)

**3. Persistencia profesional:**
   * ✅ Datos en Unity Catalog con referencias relativas
   * ✅ Scaler serializado y guardado
   * ✅ Metadatos documentados
   * ✅ Reproducible desde cualquier notebook

---

### 📈 Dataset resultante

**Tabla:** `pandito_ds.default.dl_sequences_lstm`

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `sequence_id` | INT | ID único de secuencia |
| `split` | STRING | 'train', 'validation', o 'test' |
| `target_value` | FLOAT | Valor de ventas normalizado a predecir |
| `sequence_data_b64` | STRING | Secuencia 2D serializada (12 meses × N features) |
| `timesteps` | INT | Número de pasos temporales (12) |
| `features` | INT | Número de features por paso |

**Tabla:** `pandito_ds.default.dl_metadata_lstm`

| parámetro | Descripción |
|-----------|-------------|
| `lookback` | 12 (meses históricos) |
| `forecast_horizon` | 1 (mes a predecir) |
| `n_features` | Número total de features |
| `feature_names` | Lista de nombres de features |
| `scaler_minmax` | MinMaxScaler serializado |

---

### 🚀 Próximos notebooks

**06 - Entrenamiento LSTM: (Próximamente)**
* Cargar secuencias desde UC
* Arquitectura LSTM optimizada
* Entrenamiento con EarlyStopping
* Evaluación de métricas (MAE, RMSE, MAPE)

**07 - Comparación de Arquitecturas: (Próximamente)**
* RNN vanilla vs LSTM vs GRU
* Análisis de performance
* Visualización de predicciones

---

### 💻 Cómo usar estos datos en otros notebooks

```python
import pickle
import base64
import numpy as np

# 1. Cargar tabla de secuencias
df_sequences = spark.table('pandito_ds.default.dl_sequences_lstm').toPandas()

# 2. Filtrar por split
train_df = df_sequences[df_sequences['split'] == 'train']

# 3. Deserializar secuencias
def deserialize_sequence(row):
    seq_bytes = base64.b64decode(row['sequence_data_b64'])
    return pickle.loads(seq_bytes)

X_train = np.array([deserialize_sequence(row) for _, row in train_df.iterrows()])
y_train = train_df['target_value'].values

print(f"X_train: {X_train.shape}")  # (samples, 12, features)
print(f"y_train: {y_train.shape}")  # (samples,)

# 4. Cargar scaler
metadata_df = spark.table('pandito_ds.default.dl_metadata_lstm').toPandas()
scaler_row = metadata_df[metadata_df['param_name'] == 'scaler_minmax'].iloc[0]
scaler_bytes = base64.b64decode(scaler_row['param_value'])
scaler = pickle.loads(scaler_bytes)

print("\u2705 Datos y scaler cargados desde Unity Catalog")
```

---

### 💡 Tips importantes

**⚠️ Data Leakage:**
* El scaler se ajustó SOLO con datos de entrenamiento
* La división respeta el orden temporal
* Lags y rolling features se calcularon por sucursal

**🔍 Debugging:**
* Los datos también están en `/tmp/dl_data` (temporal)
* Puedes cargar desde ahí con `np.load()` para debug rápido

**📊 Features clave:**
* Lags capturan tendencias históricas
* Rolling features suavizan ruido
* Features cíclicos evitan discontinuidades (dic → ene)
* Features geoespaciales capturan efectos de ubicación

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🧠 ¡Datos listos para Deep Learning!</h3>
  <p><i>"Feature engineering bien hecho = 80% del éxito en ML."</i></p>
</div>

## 📦 Resumen: Tablas disponibles en Unity Catalog

### 🗂️ Estructura completa del Proyecto

```
pandito_ds/                                    # Catálogo del proyecto
└── default/                                 # Schema por defecto
    ├── ventas_mensuales_mendoza_h3       # 🏭 TABLA BASE
    ├── dl_sequences_lstm                 # 🧠 SECUENCIAS LSTM
    └── dl_metadata_lstm                  # ⚙️ METADATOS
```

---

### 🏭 Tabla 1: `ventas_mensuales_mendoza_h3`

**Descripción:** Dataset base georeferenciado de Los Andes Market

**Uso:** Análisis exploratorio, visualizaciones, dashboards, ejercicios de Pandas

**Esquema:**
| Columna | Tipo | Descripción |
|---------|------|-------------|
| `fecha` | TIMESTAMP | Mes de la observación |
| `sucursal_id` | STRING | ID de sucursal (SUC001-SUC005) |
| `sucursal_nombre` | STRING | Nombre completo de la sucursal |
| `lat` | DOUBLE | Latitud GPS |
| `lon` | DOUBLE | Longitud GPS |
| `zona` | STRING | Tipo de zona comercial |
| `ventas` | DOUBLE | Ventas mensuales en $ |
| `h3_index` | STRING | Índice H3 resolución 9 (~174m) |
| `h3_res8` | STRING | Índice H3 resolución 8 (~461m) |
| `h3_res7` | STRING | Índice H3 resolución 7 (~1.22km) |

**Registros:** 300 (60 meses × 5 sucursales)

**Período:** 2019-01 a 2023-12 (60 meses)

**Cómo cargar:**
```python
# Opción 1: Completa
df = spark.table('pandito_ds.default.ventas_mensuales_mendoza_h3').toPandas()

# Opción 2: Relativa (después de USE CATALOG pandito_ds)
df = spark.table('default.ventas_mensuales_mendoza_h3').toPandas()

# Opción 3: Variables
CATALOG = "pandito_ds"
SCHEMA = "default"
df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
```

---

### 🧠 Tabla 2: `dl_sequences_lstm`

**Descripción:** Secuencias preparadas para entrenamiento de LSTM/RNN

**Uso:** Deep Learning, modelos de series temporales, forecasting

**Esquema:**
| Columna | Tipo | Descripción |
|---------|------|-------------|
| `sequence_id` | INT | ID único de secuencia |
| `split` | STRING | 'train', 'validation', o 'test' |
| `target_value` | DOUBLE | Valor normalizado a predecir |
| `sequence_data_b64` | STRING | Secuencia serializada (base64) |
| `timesteps` | INT | Número de pasos temporales (12) |
| `features` | INT | Número de features por paso |

**Registros:** 204 secuencias
* Train: 156 (70%)
* Validation: 24 (15%)
* Test: 24 (15%)

**Cómo cargar:**
```python
import pickle
import base64
import numpy as np

# Cargar secuencias
df_seq = spark.table('pandito_ds.default.dl_sequences_lstm').toPandas()

# Filtrar por split
train_df = df_seq[df_seq['split'] == 'train']

# Deserializar
def deserialize(row):
    return pickle.loads(base64.b64decode(row['sequence_data_b64']))

X_train = np.array([deserialize(row) for _, row in train_df.iterrows()])
y_train = train_df['target_value'].values

print(f"X_train shape: {X_train.shape}")  # (156, 12, N_features)
print(f"y_train shape: {y_train.shape}")  # (156,)
```

---

### ⚙️ Tabla 3: `dl_metadata_lstm`

**Descripción:** Metadatos y scaler del modelo

**Uso:** Reconstruir pipeline, desnormalizar predicciones

**Esquema:**
| Columna | Tipo | Descripción |
|---------|------|-------------|
| `param_name` | STRING | Nombre del parámetro |
| `param_value` | STRING | Valor (puede ser serializado) |
| `param_type` | STRING | Tipo de dato |

**Parámetros disponibles:**
* `lookback`: 12 (meses históricos)
* `forecast_horizon`: 1 (mes a predecir)
* `n_features`: Número de features
* `feature_names`: Lista de nombres
* `scaler_minmax`: Scaler serializado

**Cómo cargar el scaler:**
```python
import pickle
import base64

metadata = spark.table('pandito_ds.default.dl_metadata_lstm').toPandas()
scaler_row = metadata[metadata['param_name'] == 'scaler_minmax'].iloc[0]
scaler = pickle.loads(base64.b64decode(scaler_row['param_value']))

# Usar para desnormalizar
predictions_original = scaler.inverse_transform(predictions_normalized)
```

---

### 📌 Cómo usar en otros Módulos

**Módulo 06 (Agregaciones y Métricas):**
```python
# Cargar datos base
df = spark.table('pandito_ds.default.ventas_mensuales_mendoza_h3').toPandas()

# Ahora tienes datos reales para practicar GroupBy, KPIs, etc.
resumen = df.groupby('sucursal_id')['ventas'].agg(['sum', 'mean', 'count'])
```

**Módulo 07 (Series de Tiempo):**
```python
# Cargar datos base
df = spark.table('pandito_ds.default.ventas_mensuales_mendoza_h3').toPandas()
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.set_index('fecha')

# Practicar series de tiempo, forecasting, estacionalidad
```

**Módulo 09 (Geopandas):**
```python
import geopandas as gpd
from shapely.geometry import Point

# Cargar datos base
df = spark.table('pandito_ds.default.ventas_mensuales_mendoza_h3').toPandas()

# Convertir a GeoDataFrame
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Visualizar en mapa
```

**Módulo 10 (H3):**
```python
import h3

# Cargar datos base
df = spark.table('pandito_ds.default.ventas_mensuales_mendoza_h3').toPandas()

# Ya tiene columnas H3: h3_index, h3_res8, h3_res7
# Practicar operaciones H3, vecindad, agregaciones
```

---

### 🔑 Patrón estándar para todos los notebooks

**Al inicio de cada notebook:**
```python
# 1. Definir referencias
CATALOG = "pandito_ds"
SCHEMA = "default"

# 2. Usar el catálogo
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# 3. Cargar datos
df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()

print(f"✅ Datos cargados: {len(df):,} registros")
```

---

### 🚀 Ventajas de este approach

✅ **Un solo lugar de verdad:** Todos usan los mismos datos  
✅ **Consistencia:** Métricas comparables entre notebooks  
✅ **Reproducibilidad:** Ejecuta notebook maestro una vez, usa en todos  
✅ **Colaboración:** Equipo comparte datasets  
✅ **Versionado:** Delta Lake guarda historia de cambios  
✅ **Performance:** UC optimiza queries automáticamente

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🎯 ¡Notebook Maestro completado!</h3>
  <p><i>"Ahora todos los notebooks del proyecto pueden usar estas tablas."</i></p>
  <p><strong>Ejecuta este notebook primero, luego explora los demás módulos.</strong></p>
</div>

## 🎯 Resumen y próximos pasos

### Lo que hicimos:

✅ **Feature Engineering**
* Creamos 20+ features temporales (lags, rolling stats, cíclicos)
* Capturamos patrones estacionales y tendencias

✅ **Normalización correcta**
* MinMaxScaler ajustado SOLO con datos de entrenamiento
* Escalado consistente en val/test

✅ **Secuencias para LSTM**
* Formato 3D: (samples, timesteps, features)
* Ventanas deslizantes de 12 meses
* División temporal respetada

✅ **Datos listos para Deep Learning**
* Train: 21 secuencias
* Validation: 4 secuencias  
* Test: 4 secuencias

### 📚 Próximo Notebook:

**03_RNN_LSTM_Fundamentos.ipynb**
* Arquitectura de redes neuronales recurrentes
* Teoría de LSTM (Long Short-Term Memory)
* Implementación con TensorFlow/Keras
* Primeros modelos predictivos

---

👉 **Consejo**: En series temporales reales, experimentar con diferentes valores de `LOOKBACK` (6, 12, 24 meses) puede mejorar significativamente el rendimiento del modelo.